In [ ]:
# Problema: Organizar una serie temporal real en particiones Parquet locales y comprobar su recuperación.

from pathlib import Path

import pandas as pd

ROOT = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "data").is_dir() and (p / "submission").is_dir()
)
SOURCE, LAKE, OUTPUT = (
    ROOT / "data/cta_daily_station_totals.parquet",
    ROOT / "temp/lake/curated/cta_rides",
    ROOT / "submission/lake_summary.csv",
)


In [ ]:
frame = pd.read_parquet(SOURCE, engine="pyarrow")
frame["date"] = pd.to_datetime(frame["date"])
frame["year"], frame["month"] = frame.date.dt.year, frame.date.dt.month
frame.shape


In [ ]:
for file in LAKE.rglob("*.parquet") if LAKE.exists() else []:
    file.unlink()
for (year, month), part in frame.groupby(["year", "month"]):
    path = LAKE / f"year={year}" / f"month={month:02d}"
    path.mkdir(parents=True, exist_ok=True)
    part.drop(columns=["year", "month"]).to_parquet(
        path / "rides.parquet", index=False, engine="pyarrow"
    )


In [ ]:
files = list(LAKE.rglob("*.parquet"))
assert sum(len(pd.read_parquet(file)) for file in files) == len(frame)
len(files)


In [ ]:
pd.DataFrame(
    [
        [
            "cta_rides",
            "year,month",
            frame[["year", "month"]].drop_duplicates().shape[0],
            len(files),
            len(frame),
        ]
    ],
    columns=[
        "dataset",
        "partition_columns",
        "partition_count",
        "file_count",
        "row_count",
    ],
).to_csv(OUTPUT, index=False)
